# ComfyUI sur Kaggle GPU + tunnel securise
Expose une instance ComfyUI publique protegee par token (`X-Hephaistos-Token`).

**Secrets requis (Kaggle > Add-ons > Secrets) :** `HEPHAISTOS_TOKEN`

Copier l'URL cloudflared affichee en fin de cellule 4 dans `KAGGLE_COMFY_URL`.

In [ ]:
# 1. ComfyUI + dependances
import os, subprocess, sys
os.chdir('/kaggle/working')
if not os.path.exists('ComfyUI'):
    subprocess.run(['git','clone','--depth','1','https://github.com/comfyanonymous/ComfyUI.git'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','ComfyUI/requirements.txt'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','aiohttp','huggingface_hub'],check=True)
print('ComfyUI installe')

In [ ]:
# 2. Checkpoint SDXL (public, pas de token requis)
from huggingface_hub import hf_hub_download
import os
os.makedirs('/kaggle/working/ComfyUI/models/checkpoints', exist_ok=True)
p = hf_hub_download('stabilityai/stable-diffusion-xl-base-1.0','sd_xl_base_1.0.safetensors',
    local_dir='/kaggle/working/ComfyUI/models/checkpoints')
print('ckpt:', p)

In [ ]:
# 3. Proxy auth -> ComfyUI (le tunnel ne voit que le proxy token-gate)
from kaggle_secrets import UserSecretsClient
import os, threading, subprocess, sys, time, asyncio, json
from aiohttp import web, ClientSession
TOKEN = UserSecretsClient().get_secret('HEPHAISTOS_TOKEN')

comfy = subprocess.Popen([sys.executable,'main.py','--listen','127.0.0.1','--port','8188'],
    cwd='/kaggle/working/ComfyUI', stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

async def proxy(req):
    if req.headers.get('X-Hephaistos-Token') != TOKEN:
        return web.Response(status=401, text='bad token')
    async with ClientSession() as s:
        url = f'http://127.0.0.1:8188{req.path_qs}'
        body = await req.read() if req.can_read_body else None
        async with s.request(req.method, url, data=body,
            headers={k:v for k,v in req.headers.items() if k.lower() not in ('host','x-hephaistos-token')}) as up:
            return web.Response(status=up.status, body=await up.read(),
                headers={k:v for k,v in up.headers.items() if k.lower() not in ('content-encoding','content-length','transfer-encoding')})

async def main():
    app = web.Application(); app.router.add_route('*','/{path:.*}',proxy)
    r = web.AppRunner(app); await r.setup()
    await web.TCPSite(r,'0.0.0.0',8080).start()
    print('proxy pret :8080')

threading.Thread(target=lambda: asyncio.run(main()), daemon=True).start()
time.sleep(15); print('comfy pid', comfy.pid)

In [ ]:
# 4. Tunnel cloudflared -> URL publique a coller dans KAGGLE_COMFY_URL
import subprocess, re, time, threading
if not os.path.exists('/kaggle/working/cloudflared'):
    subprocess.run(['wget','-q','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64','-O','/kaggle/working/cloudflared'],check=True)
    subprocess.run(['chmod','+x','/kaggle/working/cloudflared'],check=True)
proc = subprocess.Popen(['/kaggle/working/cloudflared','tunnel','--url','http://127.0.0.1:8080'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
def pump():
    global url
    for line in proc.stdout:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m and not url:
            url = m.group(0); print('TUNNEL:', url)
threading.Thread(target=pump, daemon=True).start()
for _ in range(30):
    if url: break
    time.sleep(2)
print('
>>> KAGGLE_COMFY_URL =', url)

## Utilisation
1. Executer toutes les cellules (GPU T4x2/P100 suffit pour SDXL).
2. Copier `KAGGLE_COMFY_URL` + garder le meme `HEPHAISTOS_TOKEN` que `KAGGLE_COMFY_TOKEN` cote local.
3. Workflow local : `KaggleGPU.json` -> AgyProviderRouter provider `kaggle`.
Le kernel meurt au bout de ~9-12h : relancer = nouvelle URL a remettre dans l'env.